# PPG example
Load the bundled synthetic fixture with the public BioSignalsPlux reader, process its PPG signal directly, then process two offline segments through the loaded `Recording`.

In [ ]:
from pathlib import Path
import numpy as np
from multimodalphysiokit.io import read_biosignalsplux_hdf5
from multimodalphysiokit.processors import PPGProcessor
from multimodalphysiokit.utils import plot_signal_with_phases

data_file = Path("examples/sample_data/synthetic_multimodal_180s.hdf5")
recording = read_biosignalsplux_hdf5(data_file)
recording.signal_names()

In [ ]:
phase_intervals_seconds = np.array([[0.0, 90.0], [90.0, 180.0]])
recording.set_phases(phase_intervals_seconds, ["segment_1", "segment_2"])
plot_signal_with_phases(recording, "ppg", title="Raw PPG signal", show=True)

In [ ]:
ppg_signal = recording.get_signal("ppg")
(ppg_signal.n_samples, ppg_signal.sampling_frequency, ppg_signal.duration_seconds)

In [ ]:
processor = PPGProcessor(label_frequency_analysis=1, return_intermediates=True)
result = processor.process(ppg_signal)

In [ ]:
{
    "feature_count": len(result.features),
    "selected_features": {
        name: result.features[name]
        for name in ("ppg_bpm_mean", "ppg_pnn50", "ppg_pulse_amplitude_mean")
    },
    "valid_pulses": (result.metadata["n_valid_pulses"], result.metadata["n_pulses"]),
    "intermediate_keys": tuple(result.intermediates),
}

In [ ]:
phase_results = recording.process_phases(processor, signal_names="ppg")

In [ ]:
{
    label: {
        "feature_count": len(value.features),
        "ppg_bpm_mean": value.features["ppg_bpm_mean"],
    }
    for label, value in phase_results.items()
}